# 03 — Distance Applications

The Haversine formula is no longer just a formula — it is a tool. This notebook uses it to solve real spatial problems: finding the nearest feature, filtering by range, drawing approximate reach circles, and computing the one distance that matters most in this course.

Each section builds a pattern you will reach for repeatedly in the final project.

In [ ]:
import math

def haversine_km(p1, p2):
    """Great-circle distance between two [lon, lat] points in kilometers."""
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))

## 1. Nearest Neighbor

The nearest-neighbor problem: given a query point, find the single closest feature in a dataset.

The pattern is the same regardless of dataset size — compute distance from the query to every candidate, return the one with the minimum value.

In [ ]:
airfields = [
    {"name": "Tinker AFB",          "coords": (-97.37, 35.39)},
    {"name": "NAS Fort Worth JRB",  "coords": (-97.04, 32.85)},
    {"name": "Dyess AFB",           "coords": (-99.85, 32.42)},
    {"name": "Sheppard AFB",        "coords": (-98.51, 33.98)},
    {"name": "Vance AFB",           "coords": (-97.92, 36.34)},
    {"name": "Altus AFB",           "coords": (-99.27, 34.66)},
    {"name": "Laughlin AFB",        "coords": (-100.78, 29.36)},
    {"name": "Goodfellow AFB",      "coords": (-100.40, 31.42)},
]


def nearest(query, features):
    """Return the feature closest to query [lon, lat]."""
    return min(features, key=lambda f: haversine_km(query, f["coords"]))


def nearest_n(query, features, n):
    """Return the n closest features to query [lon, lat], sorted nearest first."""
    return sorted(features, key=lambda f: haversine_km(query, f["coords"]))[:n]


# Query: a hypothetical target coordinate
query = (-98.80, 34.20)

closest = nearest(query, airfields)
print(f"Nearest airfield: {closest['name']}")
print(f"Distance: {haversine_km(query, closest['coords']):.1f} km")

print()
print("Top 3 nearest:")
for f in nearest_n(query, airfields, 3):
    print(f"  {f['name']:<25}  {haversine_km(query, f['coords']):.1f} km")

## 2. Distance from a Map Click

In the interactive maps module, you collected clicked coordinates. Distance computation is the natural next step: every time the user clicks, compute how far that click is from each feature in the dataset and surface the result.

The pattern: capture the click coordinate, pass it to `nearest` or `nearest_n`, display the result.

In [ ]:
from ipyleaflet import Map, GeoJSON, Marker
import ipywidgets as widgets
from IPython.display import display

# Build the airfield layer
airfield_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": list(f["coords"])},
            "properties": {"name": f["name"]}
        }
        for f in airfields
    ]
}

output = widgets.Output()

m = Map(center=(33.5, -98.8), zoom=7)
m.add(GeoJSON(data=airfield_fc))

click_marker = None

def handle_click(**kwargs):
    global click_marker
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    click_coord = (lon, lat)   # convert to [lon, lat] for haversine

    # Remove previous click marker
    if click_marker:
        m.remove(click_marker)
    click_marker = Marker(location=(lat, lon))
    m.add(click_marker)

    # Find nearest airfield
    top3 = nearest_n(click_coord, airfields, 3)

    with output:
        output.clear_output()
        print(f"Click: ({lon:.4f}, {lat:.4f})")
        for f in top3:
            d = haversine_km(click_coord, f["coords"])
            print(f"  {f['name']:<25}  {d:.1f} km")

m.on_interaction(handle_click)

display(m, output)

## 3. Radius Filtering

Given a center point and a radius in kilometers, return all features within that range. This is a spatial query — the geographic equivalent of `WHERE distance <= threshold`.

In [ ]:
def within_radius(center, features, radius_km):
    """Return all features within radius_km of center [lon, lat]."""
    return [
        (f, haversine_km(center, f["coords"]))
        for f in features
        if haversine_km(center, f["coords"]) <= radius_km
    ]


center = (-98.80, 34.20)

for radius in [100, 200, 300]:
    results = within_radius(center, airfields, radius)
    names = [f["name"] for f, _ in results]
    print(f"Within {radius:>3} km ({len(results)} found): {', '.join(names) if names else 'none'}")

## 4. Draw a Distance Circle

A distance circle is an approximation of the set of all points exactly `r` km from a center. It is not a true geodesic circle — it is a polygon of `n` points, each placed at distance `r` along evenly-spaced bearings.

The approximation: step around 360° in equal increments, compute each point's offset using a flat-plane conversion, and close the ring. For small radii (say, under 500 km) this is visually indistinguishable from the true circle on a standard web map.

In [ ]:
def circle_polygon(center, radius_km, n_points=64):
    """
    Approximate a geodesic circle as a GeoJSON Polygon Feature.

    center    : [lon, lat]
    radius_km : radius in kilometers
    n_points  : number of vertices (more = smoother)
    """
    lon, lat = center
    lat_rad = math.radians(lat)

    # Degrees per km on each axis at this latitude
    deg_per_km_lat = 1.0 / 111.32
    deg_per_km_lon = 1.0 / (111.32 * math.cos(lat_rad))

    ring = []
    for i in range(n_points):
        angle = math.radians(360.0 * i / n_points)
        d_lat = math.sin(angle) * radius_km * deg_per_km_lat
        d_lon = math.cos(angle) * radius_km * deg_per_km_lon
        ring.append([lon + d_lon, lat + d_lat])

    ring.append(ring[0])   # close the polygon

    return {
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [ring]},
        "properties": {"radius_km": radius_km}
    }


# Draw 100 km, 200 km, and 300 km circles from the center point
from ipyleaflet import Map, GeoJSON

circle_styles = [
    (100, "#2a9d8f"),
    (200, "#e9c46a"),
    (300, "#e63946"),
]

m2 = Map(center=(center[1], center[0]), zoom=6)
m2.add(GeoJSON(data=airfield_fc))

for radius, color in circle_styles:
    circle = circle_polygon(center, radius)
    fc = {"type": "FeatureCollection", "features": [circle]}
    m2.add(GeoJSON(
        data=fc,
        style={"color": color, "weight": 2, "fillOpacity": 0.05}
    ))

# Mark the center
m2.add(Marker(location=(center[1], center[0])))
m2

The circles show the reach envelope. Any airfield inside the 200 km ring is within range; any outside is not. The radius filter from Section 3 and the circle from Section 4 are answering the same question — one numerically, one visually.

## 5. Missile Geometry Tie-In

Everything in this module has been building toward a concrete application: given a **launch point** and a **target point**, compute the distance between them.

In the context of this course, that distance is the **range requirement** — the minimum distance a system must be capable of to reach the target from the launch point. It is also the foundation for determining whether a target is within reach at all.

In [ ]:
launch_point  = (-98.51, 33.98)   # Sheppard AFB, TX
target_point  = (-96.80, 32.78)   # Dallas

range_km = haversine_km(launch_point, target_point)
print(f"Launch:  {launch_point}")
print(f"Target:  {target_point}")
print(f"Range:   {range_km:.1f} km")

In [ ]:
def in_range(launch, target, max_range_km):
    """Return True if target is within max_range_km of launch."""
    return haversine_km(launch, target) <= max_range_km


# Which targets are reachable from Sheppard at 300 km range?
targets = [
    {"name": "Dallas",         "coords": (-96.80, 32.78)},
    {"name": "Oklahoma City",  "coords": (-97.52, 35.47)},
    {"name": "Lubbock",        "coords": (-101.87, 33.57)},
    {"name": "San Antonio",    "coords": (-98.49, 29.42)},
    {"name": "Kansas City",    "coords": (-94.58, 39.10)},
    {"name": "El Paso",        "coords": (-106.49, 31.76)},
]

MAX_RANGE = 300   # km

print(f"Reachable targets from Sheppard AFB within {MAX_RANGE} km:\n")
for t in sorted(targets, key=lambda t: haversine_km(launch_point, t["coords"])):
    d = haversine_km(launch_point, t["coords"])
    status = "IN RANGE" if d <= MAX_RANGE else "out of range"
    print(f"  {t['name']:<18}  {d:>7.1f} km   {status}")

In [ ]:
# Visualize launch point, targets, range ring, and a line to each in-range target
from ipyleaflet import Map, GeoJSON, Marker

in_range_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": list(t["coords"])},
            "properties": {"name": t["name"], "in_range": in_range(launch_point, t["coords"], MAX_RANGE)}
        }
        for t in targets
    ]
}

lines_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [list(launch_point), list(t["coords"])]},
            "properties": {"name": t["name"], "distance_km": round(haversine_km(launch_point, t["coords"]), 1)}
        }
        for t in targets if in_range(launch_point, t["coords"], MAX_RANGE)
    ]
}

range_ring = circle_polygon(launch_point, MAX_RANGE)

m3 = Map(center=(launch_point[1], launch_point[0]), zoom=5)
m3.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [range_ring]},
    style={"color": "#e63946", "weight": 2, "fillOpacity": 0.05}
))
m3.add(GeoJSON(data=in_range_fc))
m3.add(GeoJSON(data=lines_fc, style={"color": "#2a9d8f", "weight": 2}))
m3.add(Marker(location=(launch_point[1], launch_point[0])))
m3

---

## Exercise A — Find the Nearest 5 Airfields

Given a new query point, find the 5 nearest airfields from the `airfields` list. Print each name and distance. Then display them on a map with lines from the query to each result.

In [ ]:
query_b = (-100.0, 35.0)   # somewhere in the Texas panhandle

# your code here
# 1. use nearest_n to get the 5 closest airfields
# 2. print name + distance for each
# 3. display on a map with lines from query_b to each result

## Exercise B — Radius Filter with Interactive Slider

Build an interactive map where a `IntSlider` widget controls the radius (50–500 km in 50 km steps). Each time the slider changes, re-run `within_radius` from the center point and update the output widget to show which airfields are currently in range.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.IntSlider(value=200, min=50, max=500, step=50, description="Radius km:")
out = widgets.Output()

def update_radius(change):
    with out:
        out.clear_output()
        results = within_radius(center, airfields, change["new"])
        print(f"Within {change['new']} km of {center}:")
        if results:
            for f, d in sorted(results, key=lambda x: x[1]):
                print(f"  {f['name']:<25}  {d:.1f} km")
        else:
            print("  (none)")

slider.observe(update_radius, names="value")
update_radius({"new": slider.value})   # run once on load

display(slider, out)

## Exercise C — Click-to-Range

Extend the click map from Section 2. When the user clicks, draw a 150 km range circle around the click point and highlight any airfields inside it with a different marker color. Print the count and names in the output widget.

In [ ]:
# your code here
# hint: use circle_polygon() to build the ring, GeoJSON layer to display it,
# and within_radius() to find airfields inside it
# remove and re-add the circle layer on each click

---

## Check Your Understanding

You build a range analysis tool. Given a launch point and a max range, it returns all targets within reach. A user reports that a target listed as "in range" at exactly 300 km is actually 312 km away when measured on a physical map.

**Question:** Without seeing the code, identify two distinct implementation mistakes that could produce this result — one involving the distance formula and one involving coordinate handling.

```python
# your answer here
```


---

## Next

In [04 — Performance and Batching](./04-Performance_Batching.ipynb), we cover what happens when your dataset has thousands of points — batch computation, efficient sorting, and a light introduction to NumPy for vectorized distance operations.